In [1]:
import pandas as pd 
import seaborn as sns 
taxis = sns.load_dataset("taxis") 
print(taxis.shape)
print(taxis[["pickup", "dropoff"]].dtypes) 

(6433, 14)
pickup     datetime64[us]
dropoff    datetime64[us]
dtype: object


In [2]:
par_jour = taxis.set_index("pickup").resample("D")    
nb_courses_jour = par_jour["total"].count()            
ca_jour = par_jour["total"].sum()                    
print(nb_courses_jour.head(5))                           
print(ca_jour.head(5))  

pickup
2019-02-28      1
2019-03-01    241
2019-03-02    198
2019-03-03    169
2019-03-04    171
Freq: D, Name: total, dtype: int64
pickup
2019-02-28       6.30
2019-03-01    4213.83
2019-03-02    3319.02
2019-03-03    3027.32
2019-03-04    3269.08
Freq: D, Name: total, dtype: float64


In [3]:
taxis["heure"] = taxis["pickup"].dt.hour 
print(taxis[["pickup", "heure"]].head())
courses_par_heure = taxis["heure"].value_counts()
print(courses_par_heure.head(5))

               pickup  heure
0 2019-03-23 20:21:09     20
1 2019-03-04 16:11:55     16
2 2019-03-27 17:53:01     17
3 2019-03-10 01:23:59      1
4 2019-03-30 13:27:42     13
heure
18    417
19    406
17    388
20    368
14    360
Name: count, dtype: int64


In [4]:
taxis["jour_semaine"] = taxis["pickup"].dt.day_name()
print(taxis[["pickup", "jour_semaine"]].head(3)) 
pourboire_moyen_jour = taxis.groupby("jour_semaine")["tip"].mean()
print(pourboire_moyen_jour.sort_values(ascending=False))

               pickup jour_semaine
0 2019-03-23 20:21:09     Saturday
1 2019-03-04 16:11:55       Monday
2 2019-03-27 17:53:01    Wednesday
jour_semaine
Wednesday    2.087329
Thursday     2.062177
Monday       2.027316
Tuesday      2.001370
Friday       1.980108
Sunday       1.908710
Saturday     1.815143
Name: tip, dtype: float64


In [5]:
duree = taxis["dropoff"] - taxis["pickup"]
taxis["duree_min"] = duree.dt.total_seconds() / 60
print(taxis["duree_min"].head())
print(taxis["duree_min"].describe())

q1 = taxis["duree_min"].quantile(0.25)
q3 = taxis["duree_min"].quantile(0.75)
ecart = q3 - q1
coefficient = 1.5
borne_haute = q3 + coefficient * ecart
print(q1, q3, ecart, borne_haute)

courses_longues = taxis[taxis["duree_min"] > borne_haute]
print(len(courses_longues))

nulles = taxis[taxis["duree_min"] == 0]
print(nulles[["pickup", "dropoff", "distance", "total"]].head()) 
taxis_trajets = taxis[taxis["duree_min"] > 0]
print(len(taxis), len(nulles), len(taxis_trajets))

0     6.250000
1     7.083333
2     7.400000
3    25.866667
4     9.533333
Name: duree_min, dtype: float64
count    6433.000000
mean       14.349617
std        11.643892
min         0.000000
25%         6.500000
50%        10.900000
75%        18.516667
max       107.666667
Name: duree_min, dtype: float64
6.5 18.516666666666666 12.016666666666666 36.541666666666664
358
                  pickup             dropoff  distance  total
1690 2019-03-22 06:24:14 2019-03-22 06:24:14       0.0   72.0
5493 2019-03-06 12:14:00 2019-03-06 12:14:00       0.0   10.0
5624 2019-03-11 14:04:50 2019-03-11 14:04:50       0.0    3.3
5638 2019-03-13 12:22:00 2019-03-13 12:22:00       0.0   10.0
6083 2019-03-19 15:34:00 2019-03-19 15:34:00       0.0    5.0
6433 6 6427


In [6]:
longues = taxis[taxis["duree_min"] > borne_haute]                     
print(longues.sort_values("duree_min", ascending=False)[["duree_min", "distance", "total"]].head(5))         

       duree_min  distance   total
6053  107.666667     22.17   82.36
5567   93.966667     25.51   94.80
5648   89.883333     33.46  169.70
5833   88.550000     12.79   58.80
4218   81.533333     26.92  100.49


## Courses et chiffre d'affaires par jour

Le 1er mars 2019, il y a eu **241 courses** pour un chiffre d'affaires de **4 213,83 $**, soit environ 17,5 $ par course en moyenne. Le 28 février ne compte qu'une seule course (6,30 $) : le jeu de données commence à la toute fin de cette journée.

## Courses par heure

L'heure la plus chargée est **18h**, avec **417 courses** démarrées à cette heure sur l'ensemble de la période. Elle est suivie de 19h (406 courses) et de 17h (388 courses). Les heures de pointe se situent donc entre 17h et 20h, ce qui correspond probablement à la sortie du travail et au retour à la maison. Ce chiffre compte des courses, pas de l'argent.

## Pourboire moyen par jour de la semaine

Le **mercredi** a le pourboire moyen le plus élevé, avec **2,09 $ par course**, et le **samedi** le plus bas, avec **1,82 $**. L'écart est d'environ 27 centimes : il est faible, on ne peut donc pas dire que les clients du mercredi sont beaucoup plus généreux. Le pourboire reste proche de 2 $ chaque jour de la semaine.

## Durée des courses

La durée d'une course est calculée en minutes (arrivée moins départ). La moitié centrale des courses dure entre **6,5 min** (Q1) et **18,5 min** (Q3). Avec la règle de l'IQR (Q3 + 1,5 × écart), la limite haute est d'environ **36,5 min**, et **358 courses** la dépassent.

**Choix de traitement :**
- **Courses longues : gardées.** Elles sont inhabituelles mais plausibles (distances de 12 à 33 miles, prix et vitesses cohérents). Un résultat aberrant ne veut pas dire faux.
- **Courses de 0 minute : retirées de l'analyse des durées seulement.** Leur distance est de 0 mais un prix a été encaissé : la durée est fausse, le paiement est réel. Elles restent dans `taxis` pour le chiffre d'affaires et le nombre de courses, et sont exclues de `taxis_trajets`, utilisé pour les durées.